In [23]:
import pandas as pd
print("Hello, World!")

Hello, World!


In [4]:
# Load the data
df = pd.read_csv('../data/superstore.csv')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [6]:
# Display the shape of the DataFrame
df.shape

(9800, 18)

In [7]:
# Display the summary information about the DataFrame
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
dtypes: float64(2), int6

In [26]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [72]:
from dash import Dash, html, dcc, Input, Output
import dash_ag_grid as dag
import plotly.express as px

app = Dash()

df = pd.read_csv('../data/superstore.csv')
# Convert 'Order Date' to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed')
df['Month'] = df['Order Date'].dt.to_period('M')

# Requires Dash 2.17.0 or later
app.layout = [
    # Header
    html.H1("Superstore Sales Dashboard", style={'textAlign': 'center'}),
    # Filters
    html.Div([
    dcc.Dropdown(
        id='region-filter',
        options=[{'label': r, 'value': r} for r in df['Region'].unique()],
        value=df['Region'].unique()[0]
    ),
    dcc.Dropdown(
        id = 'category-filter',
        options=[{'label': c, 'value': c} for c in df['Category'].unique()],
        value=df['Category'].unique()[0]
    ),
    dcc.DatePickerRange(
        id='date-picker',
        start_date=df['Order Date'].min(),
        end_date=df['Order Date'].max(),
        display_format='YYYY-MM-DD'
    ),
    html.Div(id='kpi-cards', style={'display': 'flex', 'gap':'20px'}),
    ], style={'display': 'flex', 'gap':'20px'},),
    
    html.Br(),
    # KPI Cards
    #html.Div(id='kpi-cards', style={'display': 'flex', 'gap':'20px'}),

    html.Br(),

    # Additional charts or tables can be added here
    html.Div([
        dcc.Graph(id='sales-trend'),
        dcc.Graph(id='category-pie')
    ], style={'display': 'flex', 'gap':'20px'}),

    html.Div([
        dcc.Graph(id='subcat-bar'),
        dcc.Graph(id='profit-region')
    ], style={'display': 'flex', 'gap':'20px'}),

    html.Br(),

    html.H3("⚠️ Loss-Making Products"),
    html.Div(id='loss-table')

]
# callback
@app.callback(
    [Output('kpi-cards', 'children'),
    Output('sales-trend', 'figure'),
    Output('category-pie', 'figure'),
    Output('subcat-bar', 'figure'),
    Output('profit-region', 'figure'),
    Output('loss-table', 'children')],
    
    [Input('region-filter', 'value'),
    Input('category-filter', 'value'),
    Input('date-picker', 'start_date'),
    Input('date-picker', 'end_date')]
)
def update_graph(selected_region, selected_category, start_date, end_date):
    # filter the DataFrame based on the selected region
    filtered_df = df.copy()
    
    if selected_region:
        filtered_df = filtered_df[filtered_df['Region'] == selected_region] 
    if selected_category:
        filtered_df = filtered_df[filtered_df['Category'] == selected_category]
    
    filtered_df = filtered_df[
        (filtered_df['Order Date'] >= start_date) &
        (filtered_df['Order Date'] <= end_date)
    ]

    # KPIs
    total_sales = filtered_df['Sales'].sum()
    #total_profit = filtered_df['Profit'].sum()
    total_orders = filtered_df.shape[0]
    #profit_ratio = round(total_profit / total_sales * 100, 2) if total_sales != 0 else 0

    kpis = [
        html.Div(f"💰 Sales: ${total_sales}", style=card_style()),
        #html.Div(f"📈 Profit: ${total_profit}", style=card_style()),
        html.Div(f"🛒 Orders: {total_orders}", style=card_style()),
        #html.Div(f"📊 Profit Ratio: {profit_ratio}%", style=card_style())
    ]    
    
    
    
    
    fig = px.line(filtered_df, x='Order Date', y='Sales', title=f'Sales Over Time - {selected_region}')
    return fig




# Card Style Function
def card_style():
    return {
        'padding': '20px',
        'border': '1px solid #ddd',
        'borderRadius': '10px',
        'boxShadow': '2px 2px 5px lightgrey',
        'fontSize': '20px'
    }

if __name__ == '__main__':
    app.run(debug=True)


[2026-04-14 15:25:12,519] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\Users\mukta\OneDrive\Documents\JobHunts\Data-Science-Projects\.venv\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mukta\OneDrive\Documents\JobHunts\Data-Science-Projects\.venv\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mukta\OneDrive\Documents\JobHunts\Data-Science-Projects\.venv\Lib\site-packages\dash\_get_app.py", line 17, in wrap
    return ctx.run(func, self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mukta\OneDrive\Documents\JobHunts\Data-Science-Projects\.venv\Lib\site-packages\dash\dash.py", 